[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/01-getting-started/02_loading_your_own_csv.ipynb)
# Loading Your Own CSV

The previous notebook used a tiny, clean, in-memory DataFrame. Real CSVs are rarely that tidy, typos creep into the *source* data itself, not just into the queries you run against it.

In this notebook you will:

1. Load a CSV with `pandas`, one with realistic typos already baked into the records
2. Inspect it for the data issues M|BOX cares about before compiling
3. Validate your setup with `TableIndexer.validate()` before building
4. Build an index and show that M|BOX can still resolve records correctly, even when the source data itself is messy


## 1. Start with a CSV

To keep this notebook self-contained, we'll write a small sample CSV to disk first, but from here on, treat this exactly like a file you'd download from your own systems. In your own project, skip this cell and point `pd.read_csv()` at your actual file.

Notice this dataset isn't clean. `full_name`, `city`, and `address` all contain small typos and inconsistencies, a missing letter, a dropped word, an inconsistent abbreviation. This is normal. Source systems are messy, and M|BOX is built to work with data exactly like this, not despite it.

In [2]:
import pandas as pd

# Simulating a real CSV export, typos and all
csv_contents = """customer_id,full_name,city,address,age,source_system
C2001,Jonathan Reevs,Chiago,742 Evergreen Terace,37,CRM_LEGACY
C2002,Priyanka Chopra,Austin,150 Riverside Drive,29,CRM_LEGACY
C2003,Muhamad Al-Farsi,Dubai,12 Sheik Zayed Rd,45,CRM_NEW
C2004,Elanor Whitfield,Dublin,9 St. Stephens Grn,52,CRM_NEW
"""

with open("datasets/customers.csv", "w", encoding="utf-8") as f:
    f.write(csv_contents)

print("customers.csv written.")

customers.csv written.


In [4]:
df = pd.read_csv("datasets/customers.csv")
df

,customer_id,full_name,city,address,age,source_system
0,C2001,Jonathan Reevs,Chiago,742 Evergreen Terace,37,CRM_LEGACY
1,C2002,Priyanka Chopra,Austin,150 Riverside Drive,29,CRM_LEGACY
2,C2003,Muhamad Al-Farsi,Dubai,12 Sheik Zayed Rd,45,CRM_NEW
3,C2004,Elanor Whitfield,Dublin,9 St. Stephens Grn,52,CRM_NEW


## 2. Inspect before you index

Before compiling, it's worth checking two things:

- **Column dtypes**, M|BOX infers indexing strategy partly from dtype (`object`/`string` → text search, `int64` → integer matching, `float64` → numeric matching). Multi-word text like `address` will be inferred as `IndexType.PHRASE`, while single-word fields like `city` typically infer as `IndexType.TERM`.
- **Forbidden characters**, string values can't contain tab (`\t`) or newline (`\n`) characters. These are used as internal delimiters during compilation, and `TableIndexer` will raise a `ValueError` if it finds them.

A quick sanity check for both:

In [6]:
print(df.dtypes)
print()

# Check for forbidden characters in any string column
has_forbidden = df.select_dtypes(include=["object", "str"]).apply(
    lambda col: col.astype(str).str.contains(r"[\t\n]")
).any().any()

print("Contains forbidden tab/newline characters:", has_forbidden)

customer_id        str
full_name          str
city               str
address            str
age              int64
source_system      str
dtype: object

Contains forbidden tab/newline characters: False


> 💡 **Also watch for:** the column name `__input_row_id__` is reserved internally by M|BOX for row tracking. If your CSV happens to have a column with that exact name, rename it before indexing.

## 3. Validate before you compile

`TableIndexer.validate()` checks your DataFrame and settings against all the structural requirements, without spending time compiling the full C-backed index. This is cheap to run and catches problems early, especially useful once this becomes part of an automated pipeline.

We'll index `full_name`, `city`, and `address`, and leave `customer_id`, `age`, and `source_system` as unindexed payload columns for now.

In [7]:
from mbox.indexing import TableIndexer

is_valid = TableIndexer.validate(
    df=df,
    index_columns=["full_name", "city", "address"]
)

print("Valid:", is_valid)

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object
Valid: True


If `validate()` finds a problem, a forbidden character, a reserved column name, an incompatible dtype, it raises a `ValueError` describing exactly what's wrong, rather than failing partway through a full compilation. In a pipeline, this is the check you'd run right after loading the CSV and before doing anything expensive.

Notice `validate()` passes here even though the *data itself* has typos. Validation checks structure, not spelling, data quality issues like "Chiago" instead of "Chicago" are exactly what the recall/matching phase is designed to handle, not something you need to clean up first.

## 4. Build the index

With validation passed, build the index exactly as before, now across three searchable fields.

In [13]:
index = TableIndexer.create_index(
    df=df,
    index_columns=["full_name", "city", "address"],
    tmp_dir= "tmp_index"
)

index

## Inspecting the compiled index

Before querying, it's often useful to see what the index actually looks like under the hood, how each field was typed, how many unique values it holds, and how much memory it's using. `TableIndex.describe()` gives you exactly that, as a DataFrame.

This is especially handy for two things: confirming M|BOX inferred the `IndexType` you expected for each column, and getting a rough sense of index size before deciding whether to save it to disk (covered in the next notebook).

In [14]:
index.describe()

,Field,Index Type,Unique Value Count,Index size [bytes]
0,full_name,IndexType.PHRASE,4,26768
1,city,IndexType.TERM,4,9508
2,address,IndexType.PHRASE,4,27696
3,customer_id,IndexType.NON_SEARCHABLE,4,0
4,age,IndexType.NON_SEARCHABLE,4,0
5,source_system,IndexType.NON_SEARCHABLE,4,0


### Reading the describe output

| Column | Meaning |
|---|---|
| `Field` | The column name from your original DataFrame |
| `Index Type` | The `IndexType` M|BOX inferred (or that you explicitly set) for this field |
| `Unique Value Count` | How many distinct values exist for this field across the indexed rows |
| `Index size [bytes]` | Memory footprint of this field's compiled search structure |

A few things to notice in this result:

- `full_name` and `address` were both inferred as `IndexType.PHRASE` — makes sense, since both are multi-word strings with spaces. `city` was inferred as `IndexType.TERM`, since city names are typically single words.
- `customer_id`, `age`, and `source_system` show up as `IndexType.NON_SEARCHABLE` with `0` bytes of index size — they weren't passed to `index_columns`, so no search structure was built for them at all. They're stored as payload, not indexed.
- `address` has a slightly larger index size than `full_name` here, despite the same unique value count, a reminder that index size scales with the actual text content of a field, not just row count.

Notice `customer_id`, `age`, and `source_system` weren't passed to `index_columns`, they're still attached to each record and will show up in results as payload, but they aren't searched against.

## 5. Query your real data

Here's the interesting part. Let's search using the **correctly spelled** name, city, and address for our first record, even though the source row itself was stored as `"Jonathan Reevs"`, `"Chiago"`, `"742 Evergreen Terace"`.

A naive exact match against this source data would fail, because the record was *never* stored correctly in the first place. M|BOX resolves it anyway.

In [10]:
results = index.match(
    full_name="Jonathan Reeves",
    city="Chicago",
    address="742 Evergreen Terace",
    include_field_scores=True
)

results

,query_row,index_row,full_name_candidate,city_candidate,address_candidate,customer_id_candidate,age_candidate,source_system_candidate,overall_score,full_name_score,city_score,address_score
0,0,0,Jonathan Reevs,Chiago,742 Evergreen Terace,C2001,37,CRM_LEGACY,84,83,69,100


Even with typos on *both* sides, a correctly spelled query against a misspelled source record, M|BOX still surfaces the right candidate, with field-level scores showing exactly how confident each part of the match was. This is the core value of fault-tolerant matching: it doesn't assume either your queries or your source data are clean.

Notice `customer_id_candidate`, `age_candidate`, and `source_system_candidate` also showed up in the results, even though we only searched on `full_name`, `city`, and `address`. This is the payload behavior mentioned earlier: unindexed columns aren't matched against, but they travel with the record and are returned alongside every result, so you get the full row back, not just the fields you searched on.

## Next steps

- **`03_saving_and_loading_indexes.ipynb`**, compile once, save to disk, and skip rebuilding the index on every run
- **`2-data-harmonization/`**, handle accents, umlauts, and nicknames automatically with `CharacterMapping` and `AliasSet`
- **`6-production/validating_before_you_compile.ipynb`**, a deeper look at `TableIndexer.validate()` for production pipelines

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*